# Lia: distância de edição para antigolpe e busca tolerante a erro

**Global Solution 2026.2 · CitizenTech · Dynamic Programming · Equipe AuxiLia**

| Integrante | RM |
|---|---|
| João Victor Melo Santos | 566640 |
| Murilo Jeronimo Ferreira Nunes | 560641 |
| Vinicius Kozonoe Guaglini | 567264 |
| Yan Lucas Gonçalves da Silva | 567046 |

## 1. O problema

A lia é um assistente de voz para pessoas com pouca prática de leitura e de cecular, em Taiobeiras (MG). Duas situações do dia a dia dela pedem o mesmo algoritmo:
1. **Golpe por link parecido.** A pessoa recebe um link como `g0v.br` ou `caixa-gov.br`, quase igual ao endereço oficial. Para quem lê com dificuldade, a diferença de uma letra não aparece.
2. **Pedido com erro.** A pessoa fala "benefisio do inss" ou "agendar consuta", e o texto chega com erro. A Lia precisa entender mesmo assim.

Nos dois casos, a pergunta é a mesma: **quão parecidos são dois textos?** A resposta é a distância de edição (Levenshtein), calculada com programação dinâmica.

## 2. Os dados

- `dados/dominios_oficiais.csv`: endereços oficiais de órgãos públicos, bancos e serviços. São dados públicos, conferidos nos sites oficiais em 16/11/2026.
- `dados/jeitos_de_pedir.csv`: frases que as pessoas usam para pedir cada ação da Lia. São dados simulados pela equipe. Não há nenhum dado pessoal.

In [1]:
import csv
import unicodedata

def carregar(caminho):
    """Lê um arquivo CSV e devolve uma lista de dicionários, um por linha."""
    with open(caminho, encoding="utf-8") as arquivo:
        return list(csv.DictReader(arquivo))

def normalizar(texto):
    """Deixa o texto em minúsculas, sem espaços nas pontas e sem acentos."""
    texto = texto.lower().strip()
    separado = unicodedata.normalize('NFD', texto)
    return "".join(letra for letra in separado if unicodedata.category(letra) != "Mn")

dominios = carregar("dados/dominios_oficiais.csv")
jeitos = carregar("dados/jeitos_de_pedir.csv")

print(len(dominios), "domínios oficiais")
print(len(jeitos), "jeitos de pedir")
print(dominios[0])
print(normalizar("  Benefício do INSS "))

15 domínios oficiais
23 jeitos de pedir
{'dominio': 'gov.br', 'servico': 'Portal gov.br', 'tipo': 'governo'}
beneficio do inss


## 3. O algoritmo: distância de Levenshtein com programação dinâmica

A distância de Levenshtein entre dois textos é o menor número de operações para um virar o outro. As operações são trocar, inserir ou apagar uma letra.

**Estrutura de dados:** uma tabela `t` com `(m + 1)` linhas e `(n + 1)` colunas, em que `m` e `n` são os tamanhos dos textos. A casa `t[i][j]` guarda a distância entre as `i` primeiras letras de `a` e as `j` primeiras letras de `b`.

**Regra de preenchimento:**

- `t[i][0] = i` e `t[0][j] = j` (comparação com o texto vazio);
- `t[i][j] = min(t[i-1][j] + 1, t[i][j-1] + 1, t[i-1][j-1] + custo)`, com `custo = 0` se `a[i-1] == b[j-1]` e `custo = 1` se não.

Cada casa é calculada uma vez só, a partir de três casas já prontas. É isso que faz ser programação dinâmica: os subproblemas se repetem, e a tabela evita recalcular.

In [5]:
def tabela_de_distancias(a, b):
    """Monta a tabela de PD: t[i][j] é a distância entre as i primeiras letras de a e as j primeiras de b."""
    m, n = len(a), len(b)
    t = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(m + 1):
        t[i][0] = i
    for j in range(n + 1):
        t[0][j] = j

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            custo = 0 if a[i - 1] == b[j - 1] else 1
            t[i][j] = min(
                t[i - 1][j] + 1,
                t[i][j - 1] + 1,
                t[i - 1][j - 1] + custo
            )
    return t

def distancia(a, b):
    """Distância de Levenshtein: o menor número de trocas, inserções e remoções para a virar b."""
    t = tabela_de_distancias(a, b)
    return t[len(a)][len(b)]

def mostrar_tabela(a, b):
    """Desenha a tabela de PD, com as letras de a nas linhas e as de b nas colunas."""
    t = tabela_de_distancias(a, b)
    print("      " + "  ".join(b))
    for i, linha in enumerate(t):
        letra = a[i - 1] if i > 0 else " "
        print(letra, " ".join(f"{valor:2}" for valor in linha))

mostrar_tabela("casa", "caso")

      c  a  s  o
   0  1  2  3  4
c  1  0  1  2  3
a  2  1  0  1  2
s  3  2  1  0  1
a  4  3  2  1  1


In [6]:
testes = [
    ("casa", "casa", 0),
    ("casa", "caso", 1),
    ("gato", "pato", 1),
    ("inss", "ins", 1),
    ("", "abc", 3),
    ("beneficio", "benefisio", 1),
    ("kitten", "sitting", 3),
]

for a, b, esperado in testes:
    obtido = distancia(a, b)
    assert obtido == esperado, f"{a} -> {b}: esperado {esperado}, obtido {obtido}"
    print (f"{a!r} -> {b!r}: {obtido}")

print ("Todos os testes passaram.")

'casa' -> 'casa': 0
'casa' -> 'caso': 1
'gato' -> 'pato': 1
'inss' -> 'ins': 1
'' -> 'abc': 3
'beneficio' -> 'benefisio': 1
'kitten' -> 'sitting': 3
Todos os testes passaram.
